In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube
from astropy.coordinates import SkyCoord
import sys
import plotfancy as pf
from matplotlib.patches import Circle
from astropy.visualization import ZScaleInterval
from astropy.table import Table
from types import SimpleNamespace

In [ ]:
def generate_gaussian_profile(params, num_points=500, width_factor=4.0):
    sigma = params.fwhm / (2 * np.sqrt(2 * np.log(2)))
    x_min = params.lpeak - width_factor * params.fwhm / 2
    x_max = params.lpeak + width_factor * params.fwhm / 2
    x = np.linspace(x_min, x_max, num_points)
    y = params.cont + params.peak * np.exp(-((x - params.lpeak) ** 2) / (2 * sigma ** 2))
    return np.vstack((x, y))

def load_cube(path):
    print(f'Loading datacube: {path}...')
    try:
        cube = Cube(path)
        print(f'Successfully loaded cube. Dimensions: {cube.shape}')
        return cube
    except FileNotFoundError:
        print(f'Error: Datacube not found at {path}')
        sys.exit(1)

def extract_spectrum(cube, ra, dec, radius):
    center = (dec, ra)
    print(f'\nExtracting spectrum at (RA, Dec) = ({ra:.6f}, {dec:.6f}) with a {radius}" radius aperture.')
    spec = cube.aperture(center, radius, is_sum=True)
    print('Extraction complete.')
    return spec

def measure_redshift(spec, z_guess):
    print('\nMeasuring spectroscopic redshift...')
    lines_for_z = {'Hbeta': 4861.33, 'OIII_5007': 5006.84, 'Halpha': 6562.80}
    redshifts = []
    for name, rest_wave in lines_for_z.items():
        obs_wave_guess = rest_wave * (1 + z_guess)
        try:
            fit = spec.gauss_fit(lmin=(obs_wave_guess - 30), lmax=(obs_wave_guess + 30), plot=False)
            line_z = (fit.lpeak / rest_wave) - 1
            redshifts.append(line_z)
            print(fr'  - {name}: Found at {fit.lpeak:.2f} \AA, z = {line_z:.5f}')
        except Exception:
            print(fr'  - {name}: Fit failed near {obs_wave_guess:.2f} \AA.')

    if not redshifts:
        print('\nError: Could not measure redshift. Using initial guess.')
        return z_guess, 0.0
    
    z_measured = np.mean(redshifts)
    z_err = np.std(redshifts) / np.sqrt(len(redshifts)) if len(redshifts) > 1 else 0.0
    print(f'\nMeasured Redshift z = {z_measured:.5f} \xB1 {z_err:.5f}')
    return z_measured, z_err

def deredshift_spectrum(spec, z):
    spec_rest = spec.copy()
    spec_rest.wave.set_crval(spec_rest.wave.get_crval() / (1 + z))
    spec_rest.wave.set_step(spec_rest.wave.get_step() / (1 + z))
    return spec_rest

def fit_emission_lines(spec_rest):
    from types import SimpleNamespace
    lines = {'Hbeta': 4861.33, 'OIII_5007': 5006.84, 'Halpha': 6562.80, 'NII_6583': 6583.45}
    line_fits = {}
    for name, wave in lines.items():
        try:
            line_fits[name] = spec_rest.gauss_fit(lmin=(wave - 15), lmax=(wave + 15), plot=False)
        except Exception:
            sub_spec = spec_rest.subspec(lmin=wave - 15, lmax=wave + 15)
            
            spec_for_continuum = None
            if sub_spec is not None and len(sub_spec.shape) == 1 and sub_spec.shape[0] > 2:
                spec_for_continuum = sub_spec
            elif spec_rest is not None and len(spec_rest.shape) == 1 and spec_rest.shape[0] > 2:
                spec_for_continuum = spec_rest

            if spec_for_continuum:
                continuum = np.mean(spec_for_continuum.data)
                std_err = np.std(spec_for_continuum.data)
                
                fit_mock = SimpleNamespace(
                    flux=0.0,
                    err_flux=std_err,
                    peak=0.0,
                    cont=continuum,
                    lpeak=wave,
                    fwhm=1.0,
                    err_peak=std_err,
                    err_cont=std_err,
                    err_lpeak=0.0,
                    err_fwhm=0.0
                )
                line_fits[name] = fit_mock
            else:
                line_fits[name] = None
    return line_fits


In [ ]:
class museCube:
    def __init__(self, path:str, zapprox:np.float64):
        ### attributes ###
        self.path = path
        self.zapprox = zapprox # redshift guess
        ### initialisers ###
        self.init_cube(self.path)
        self.init_table()
        self.init_lambda()

    ### INITIALISERS ###

    def init_cube(self, PATH):
        print(f'Loading datacube: {PATH}...')
        try:
            cube = Cube(PATH)
            print(f'Successfully loaded cube. Dimensions: {cube.shape}')
            self.cube = cube
            return True
        except FileNotFoundError:
            print(f'Error: Datacube not found at {PATH}')
            sys.exit(1)
    
    def init_lambda(self):
        self.rest_lambdas = {
        # --- Primary [OIII] and [OII] ---
        'oiii5007': 5006.84,
        'oiii4959': 4958.91,
        'oii3726':  3726.03,
        'oii3729':  3728.82,

        # --- Hydrogen Balmer Series ---
        'halpha':   6562.80,
        'hbeta':    4861.33,
        'hgamma':   4340.46,
        'hdelta':   4101.73,

        # --- Key Diagnostic Lines ---
        'oiii4363': 4363.21,  # Auroral line for Te
        'neiii':    3868.75,

        # --- Low-Ionization Lines ---
        'nii6583':  6583.45,
        'nii6548':  6548.05,
        'sii6716':  6716.44,
        'sii6731':  6730.82,

        # --- Helium Lines ---
        'heii4686': 4685.68,
        'hei5876':  5875.62,
        }
    
    def init_table(self, cnames=None):
        column_names = ['object_id', 'ra', 'dec','z'] if cnames==None else cnames
        meta = ['flux', 'flux_err', 'ew', 'ew_err', 'centroid', 'fwhm']
        for key, _ in self.rest_lambdas.items():
            for m in meta:
                column_names.append(key+m)

        self.ex_table = Table(names=column_names,dtype=([str]+(len(column_names)-1)*[np.float64]))
        return True
    ### METHODS ###

    def extract_spectrum(self, ra, dec, radius):
        center = (dec, ra)
        print(f'\nExtracting spectrum at (RA, Dec) = ({ra:.6f}, {dec:.6f}) with a {radius}" radius aperture.')
        spec = self.cube.aperture(center, radius, is_sum=True)
        print('Extraction complete.')
        return spec
    
    def deredshift_spectrum(self, spec, z):
        spec_rest = spec.copy()
        spec_rest.wave.set_crval(spec_rest.wave.get_crval() / (1 + z))
        spec_rest.wave.set_step(spec_rest.wave.get_step() / (1 + z))
        return spec_rest
    
    def generate_gaussian_profile(self, params, num_points=500, width_factor=4.0):
        sigma = params.fwhm / (2 * np.sqrt(2 * np.log(2)))
        x_min = params.lpeak - width_factor * params.fwhm / 2
        x_max = params.lpeak + width_factor * params.fwhm / 2
        x = np.linspace(x_min, x_max, num_points)
        y = params.cont + params.peak * np.exp(-((x - params.lpeak) ** 2) / (2 * sigma ** 2))
        return np.vstack((x, y))
    
    def find_o3_5007(self, spec):
        a=1


        #     for name, rest_wave in lines_for_z.items():
        # obs_wave_guess = rest_wave * (1 + z_guess)
        # try:
        #     fit = spec.gauss_fit(lmin=(obs_wave_guess - 30), lmax=(obs_wave_guess + 30), plot=False)
        #     line_z = (fit.lpeak / rest_wave) - 1
        #     redshifts.append(line_z)
        #     print(fr'  - {name}: Found at {fit.lpeak:.2f} \AA, z = {line_z:.5f}')
        # except Exception:
        #     print(fr'  - {name}: Fit failed near {obs_wave_guess:.2f} \AA.')

NameError: name 'self' is not defined

In [18]:
CUBE_PATH = '../../cubes/s780_COMBINED_CUBE_MED_FINAL.fits'
TITLE = 'MACS'
RADIUS_ARCSEC = .6
Z_GUESS = 0.249

coords = SkyCoord('01h59m04.03s', '-34d13m31.8s', frame='icrs')
RA_DEG = coords.ra.deg
DEC_DEG = coords.dec.deg


In [15]:
test = museCube(path=CUBE_PATH)

Loading datacube: ../../cubes/s780_COMBINED_CUBE_MED_FINAL.fits...
Successfully loaded cube. Dimensions: (3682, 315, 361)
